# 05: Evaluate LLaVA + LoRA (H3 Ablation)

Loads the LoRA adapter trained in notebook 04 onto the LLaVA backbone and
runs ORA on the synthetic held-out set + a Mind2Web slice. Prints a base-vs-
adapter delta table for the H3 row of the report.

**Runtime:** A100 GPU. **Wallclock:** ~30-40 min. **Compute units:** ~7-9.

In [ ]:
# Mount Google Drive. The first run prompts for OAuth; subsequent runs
# in the same session reuse the token automatically.
from google.colab import drive
drive.mount('/content/drive')

import os, pathlib
ROOT = pathlib.Path('/content/drive/MyDrive/grounded_vla')
ROOT.mkdir(parents=True, exist_ok=True)
print('drive root:', ROOT)

In [ ]:
# Verify accelerator. Pro+ should give you A100 most of the time; if you
# see T4, change Runtime -> Change runtime type -> A100 GPU and re-run.
import subprocess
subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,memory.free',
                '--format=csv'])

In [ ]:
# Clone repo into ephemeral /content (fast scratch space).
REPO_URL = 'https://github.com/<your-org>/grounded_vla.git'
import os, subprocess
if not os.path.exists('/content/grounded_vla'):
    subprocess.run(['git', 'clone', REPO_URL, '/content/grounded_vla'], check=True)
%cd /content/grounded_vla

In [ ]:
# Install repo + GPU stack. Quiet to keep the cell output sane.
!pip install -q -e .[gpu,data]

In [ ]:
import os
WEIGHTS = '/content/drive/MyDrive/grounded_vla/hf_cache'
DATA    = '/content/drive/MyDrive/grounded_vla/data'
ADAPTER = '/content/drive/MyDrive/grounded_vla/checkpoints/llava-lora-r1'
os.environ['HF_HOME'] = WEIGHTS
os.environ['TRANSFORMERS_OFFLINE'] = '1'

In [ ]:
# Load LLaVA + adapter via a thin subclass that injects PEFT after load.
from grounded_vla.backends.llava import LLaVABackend

class LLaVAWithLoRA(LLaVABackend):
    name = 'llava-1.6-7b+lora'
    def __init__(self, adapter_dir, **kw):
        super().__init__(**kw)
        self.adapter_dir = adapter_dir
    def _ensure_loaded(self):
        if self._model is not None:
            return
        super()._ensure_loaded()
        from peft import PeftModel
        self._model = PeftModel.from_pretrained(self._model, self.adapter_dir)
        self._model.eval()
        print(f'[LLaVAWithLoRA] adapter loaded from {self.adapter_dir}')

In [ ]:
from grounded_vla.agents import ORAAgent
from grounded_vla.backends.base import GenerationConfig
from grounded_vla.data import make_dataset
from grounded_vla.eval import EvalRunner
from pathlib import Path

backend = LLaVAWithLoRA(
    adapter_dir=ADAPTER,
    model_id=f'{WEIGHTS}/llava-v1.6-mistral-7b-hf',
    device='cuda',
    quantize='4bit',
)
agent = ORAAgent(backend, GenerationConfig(max_new_tokens=384, temperature=0.1))

RUNS = Path('/content/drive/MyDrive/grounded_vla/runs')

def run(ds_name, dataset):
    out = RUNS / f'ora_lora__{ds_name}'
    res = EvalRunner(agent).evaluate(
        dataset, save_dir=out, checkpoint_every=5, resume=True
    )
    print(f'ora_lora on {ds_name}: '
          f'success={res.task_completion_rate:.3f} '
          f'mean_steps={res.mean_steps:.2f} '
          f'errors={res.error_breakdown}')
    return res

synth = make_dataset({
    'kind': 'jsonl',
    'path':  f'{DATA}/synthetic_sample/synthetic_sample.jsonl',
    'source': 'synthetic',
})
import glob
_m2w_jsonls = sorted(glob.glob(f'{DATA}/mind2web/*.jsonl'))
if not _m2w_jsonls:
    raise FileNotFoundError(f'no Mind2Web JSONL under {DATA}/mind2web/')
m2w = make_dataset({
    'kind': 'mind2web',
    'jsonl_path': _m2w_jsonls[0],
    'images_dir':  f'{DATA}/mind2web/images',
    'limit': 30,
})

_ = run('synthetic', synth)
_ = run('mind2web',  m2w)
backend.close()

In [ ]:
# Side-by-side vs the non-LoRA ORA runs from notebook 03.
import json, glob
base_runs = sorted(glob.glob('/content/drive/MyDrive/grounded_vla/runs/ora_llava__*'))
lora_runs = sorted(glob.glob('/content/drive/MyDrive/grounded_vla/runs/ora_lora__*'))

def load(p):
    return json.loads(open(f'{p}/summary.json').read())

print(f'{"dataset":12s} {"ora-base":>10s} {"ora+lora":>10s} {"delta":>8s}')
for lp in lora_runs:
    ds = lp.rsplit('__', 1)[-1]
    base = next((p for p in base_runs if p.endswith(ds)), None)
    if not base:
        continue
    b = load(base)['task_completion_rate']
    l = load(lp)['task_completion_rate']
    print(f'{ds:12s} {b:10.3f} {l:10.3f} {l-b:+8.3f}')

## Interpreting the H3 result

- **Positive delta on synthetic, smaller on Mind2Web:** LoRA closed the in-domain
  gap; cross-domain transfer is partial. Useful talking point in the report.
- **Negative delta:** likely overfitting on a tiny synthetic set. Either grow the
  synthetic corpus to ~200 reviewed examples, or shorten training (1 epoch).
- **No change:** LoRA rank may be too small (try `r=32`) or the action format in
  the synthetic dataset doesn't match the agent's prompt template.

Either way, the result goes into the H3 row of the results table.